# Marginal Ordinal Regression: Life Satisfaction, Job Satisfaction, and Financial Satisfaction

This notebook implements marginal ordinal regression models to predict three satisfaction outcomes:
- **LIFENOW**: Life satisfaction (1-10 scale)
- **SATJOB**: Job satisfaction (1-4 scale, where 1=very dissatisfied, 4=very satisfied)
- **SATFIN**: Financial satisfaction (1-3 scale, where 1=not satisfied, 3=pretty well satisfied)

## Modeling Approach

We fit **separate marginal ordinal probit models** for each outcome, then estimate the residual correlation structure between outcomes using polychoric correlations. This approach:

1. Models each outcome independently using ordinal probit regression
2. Estimates polychoric correlations to understand shared latent structure
3. Fits a combined model with correlation matrix estimation via LKJ prior

Each outcome is modeled as an ordinal manifestation of an underlying continuous latent variable:

$$
Y_k^* = X'\beta_k + \varepsilon_k \quad \text{for } k = 1, 2, 3
$$

The observed ordinal category is determined by thresholds:
$$
Y_k = j \quad \text{if} \quad \theta_{k,j-1} < Y_k^* \leq \theta_{k,j}
$$

**Note:** While we estimate a correlation matrix to characterize the association between outcomes, the likelihood is computed marginally for each outcome. This is distinct from a fully joint multivariate ordinal model, which would require computing multivariate normal rectangle probabilities.

In [1]:
import pandas as pd
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.special import ndtr  # Standard normal CDF
from pathlib import Path

np.random.seed(42)
pd.set_option('display.max_columns', None)

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

PyMC version: 5.27.0
ArviZ version: 0.22.0


In [2]:
# Regularized Horseshoe Prior Helper Function (Piironen & Vehtari 2017)

def horseshoe_prior(name, D, N, D0=3, slab_scale=2.5, slab_df=4, dims=None):
    """Regularized horseshoe prior following Piironen & Vehtari (2017).
    
    This prior provides adaptive shrinkage: small effects are shrunk toward zero
    while large effects are preserved. The slab component prevents completely
    unregularized coefficients.
    
    Args:
        name: Base name for the prior components
        D: Number of predictors
        N: Sample size
        D0: Expected number of non-zero coefficients (prior sparsity belief)
        slab_scale: Scale for the slab component (default 2.5)
        slab_df: Degrees of freedom for slab (default 4)
        dims: Optional dimension name for the coefficients
    
    Returns:
        beta: Deterministic node containing the regularized coefficients
    
    Prior structure:
        tau ~ HalfStudentT(nu=2, sigma=tau0)  # global shrinkage
        lambda_j ~ HalfStudentT(nu=5, sigma=1)  # local shrinkage
        c2 ~ InverseGamma(slab_df/2, slab_df/2 * slab_scale^2)  # slab regularization
        
        lambda_tilde_j = lambda_j * sqrt(c2 / (c2 + tau^2 * lambda_j^2))
        z_j ~ Normal(0, 1)
        beta_j = z_j * tau * lambda_tilde_j
    """
    # Global scale based on expected sparsity (Piironen & Vehtari formula)
    tau0 = D0 / (D - D0) / np.sqrt(N)
    
    # Global shrinkage parameter
    tau = pm.HalfStudentT(f'{name}_tau', nu=2, sigma=tau0)
    
    # Local shrinkage parameters (one per predictor)
    if dims is not None:
        lam = pm.HalfStudentT(f'{name}_lam', nu=5, sigma=1, dims=dims)
        z = pm.Normal(f'{name}_z', 0, 1, dims=dims)
    else:
        lam = pm.HalfStudentT(f'{name}_lam', nu=5, sigma=1, shape=D)
        z = pm.Normal(f'{name}_z', 0, 1, shape=D)
    
    # Slab component (regularizes large coefficients)
    c2 = pm.InverseGamma(f'{name}_c2', alpha=slab_df/2, beta=slab_df/2 * slab_scale**2)
    
    # Regularized local shrinkage
    lam_tilde = lam * pt.sqrt(c2 / (c2 + tau**2 * lam**2))
    
    # Non-centered parameterization for better sampling
    if dims is not None:
        beta = pm.Deterministic(name, z * tau * lam_tilde, dims=dims)
    else:
        beta = pm.Deterministic(name, z * tau * lam_tilde)
    
    return beta

print("Horseshoe prior helper function defined.")

Horseshoe prior helper function defined.


## Data Loading and Preparation

In [3]:
# Load data
df = pd.read_csv('../data/gss_2022.csv')
print(f'Full dataset: {df.shape}')

# Define outcomes and predictors
# Note: satfin is now an outcome, not a predictor
outcomes = ['lifenow', 'satjob', 'satfin']
predictors = ['hlthdep', 'stress', 'wrkmeangfl', 'finrela', 'anxiety', 'age', 'sex', 'degree',
              'hours_worked', 'race', 'relig', 'wrkstat', 'chngtme']

# Select columns and drop missing
analysis_cols = outcomes + predictors
df_analysis = df[analysis_cols].dropna().copy()

# Filter to valid outcome values
df_analysis = df_analysis[
    (df_analysis['lifenow'].between(1, 10)) & 
    (df_analysis['satjob'].between(1, 4)) &
    (df_analysis['satfin'].between(1, 3))
].copy()

print(f'Analysis dataset: {df_analysis.shape}')
print(f'Complete cases: {len(df_analysis)}')

Full dataset: (3544, 27)
Analysis dataset: (479, 16)
Complete cases: 479


In [4]:
# Examine outcome distributions
print('=== LIFENOW (Life Satisfaction) ===')
print(df_analysis['lifenow'].value_counts().sort_index())
print(f'\nRange: {df_analysis["lifenow"].min():.0f} - {df_analysis["lifenow"].max():.0f}')

print('\n=== SATJOB (Job Satisfaction) ===')
print(df_analysis['satjob'].value_counts().sort_index())
print('(1=Very Dissatisfied, 2=Little Dissatisfied, 3=Mod Satisfied, 4=Very Satisfied)')

print('\n=== SATFIN (Financial Satisfaction) ===')
print(df_analysis['satfin'].value_counts().sort_index())
print('(1=Not At All Satisfied, 2=More or Less Satisfied, 3=Pretty Well Satisfied)')

=== LIFENOW (Life Satisfaction) ===
lifenow
3.0       4
4.0       8
5.0      30
6.0      49
7.0      44
8.0     124
9.0     159
10.0     61
Name: count, dtype: int64

Range: 3 - 10

=== SATJOB (Job Satisfaction) ===
satjob
1.0     16
2.0     47
3.0    202
4.0    214
Name: count, dtype: int64
(1=Very Dissatisfied, 2=Little Dissatisfied, 3=Mod Satisfied, 4=Very Satisfied)

=== SATFIN (Financial Satisfaction) ===
satfin
1.0    137
2.0    252
3.0     90
Name: count, dtype: int64
(1=Not At All Satisfied, 2=More or Less Satisfied, 3=Pretty Well Satisfied)


In [5]:
# Pairwise joint distributions and correlations
outcome_pairs = [('lifenow', 'satjob'), ('lifenow', 'satfin'), ('satjob', 'satfin')]
pair_labels = [
    ('LIFENOW', 'SATJOB'),
    ('LIFENOW', 'SATFIN'),
    ('SATJOB', 'SATFIN')
]

for (var1, var2), (label1, label2) in zip(outcome_pairs, pair_labels):
    crosstab = pd.crosstab(df_analysis[var2], df_analysis[var1])
    print(f'Joint distribution ({label2} rows × {label1} columns):')
    print(crosstab)
    
    spearman_corr = df_analysis[[var1, var2]].corr(method='spearman').iloc[0, 1]
    print(f'Spearman correlation: {spearman_corr:.3f}\n')

# Overall correlation matrix
print('=== Spearman Correlation Matrix (Outcomes) ===')
outcome_corr = df_analysis[outcomes].corr(method='spearman')
print(outcome_corr.round(3))

Joint distribution (SATJOB rows × LIFENOW columns):
lifenow  3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0
satjob                                                 
1.0         1     2     1     5     1     3     3     0
2.0         2     1     6     7    10    12     5     4
3.0         1     2    15    22    23    55    69    15
4.0         0     3     8    15    10    54    82    42
Spearman correlation: 0.303

Joint distribution (SATFIN rows × LIFENOW columns):
lifenow  3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0
satfin                                                 
1.0         3     7    13    33    23    36    14     8
2.0         1     0    13    16    20    71   100    31
3.0         0     1     4     0     1    17    45    22
Spearman correlation: 0.445

Joint distribution (SATFIN rows × SATJOB columns):
satjob  1.0  2.0  3.0  4.0
satfin                    
1.0      11   21   59   46
2.0       5   21  112  114
3.0       0    5   31   54
Spearman correlation: 0.217

=== Spe

In [6]:
# Visualize pairwise joint distributions
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'LIFENOW × SATJOB', 'LIFENOW × SATFIN', 'SATJOB × SATFIN'
])

# LIFENOW × SATJOB
crosstab1 = pd.crosstab(df_analysis['satjob'], df_analysis['lifenow'], normalize='all')
fig.add_trace(go.Heatmap(
    z=crosstab1.values,
    x=crosstab1.columns.astype(int),
    y=crosstab1.index.astype(int),
    colorscale='Blues',
    showscale=False
), row=1, col=1)

# LIFENOW × SATFIN
crosstab2 = pd.crosstab(df_analysis['satfin'], df_analysis['lifenow'], normalize='all')
fig.add_trace(go.Heatmap(
    z=crosstab2.values,
    x=crosstab2.columns.astype(int),
    y=crosstab2.index.astype(int),
    colorscale='Blues',
    showscale=False
), row=1, col=2)

# SATJOB × SATFIN
crosstab3 = pd.crosstab(df_analysis['satfin'], df_analysis['satjob'], normalize='all')
fig.add_trace(go.Heatmap(
    z=crosstab3.values,
    x=crosstab3.columns.astype(int),
    y=crosstab3.index.astype(int),
    colorscale='Blues',
    showscale=True
), row=1, col=3)

fig.update_xaxes(title_text='Life Satisfaction', row=1, col=1)
fig.update_yaxes(title_text='Job Satisfaction', row=1, col=1)
fig.update_xaxes(title_text='Life Satisfaction', row=1, col=2)
fig.update_yaxes(title_text='Financial Satisfaction', row=1, col=2)
fig.update_xaxes(title_text='Job Satisfaction', row=1, col=3)
fig.update_yaxes(title_text='Financial Satisfaction', row=1, col=3)

fig.update_layout(
    title='Pairwise Joint Distributions of Satisfaction Outcomes',
    height=350, width=1000
)
fig.show()

## Data Preparation for Modeling

In [7]:
# Prepare outcomes (0-indexed for PyMC)
df_analysis['y1'] = (df_analysis['lifenow'] - 1).astype(int)  # 0-9
df_analysis['y2'] = (df_analysis['satjob'] - 1).astype(int)   # 0-3
df_analysis['y3'] = (df_analysis['satfin'] - 1).astype(int)   # 0-2

K1 = 10  # LIFENOW categories
K2 = 4   # SATJOB categories
K3 = 3   # SATFIN categories

print(f'LIFENOW coded: 0-{K1-1} ({K1} categories, {K1-1} cutpoints)')
print(f'SATJOB coded: 0-{K2-1} ({K2} categories, {K2-1} cutpoints)')
print(f'SATFIN coded: 0-{K3-1} ({K3} categories, {K3-1} cutpoints)')

LIFENOW coded: 0-9 (10 categories, 9 cutpoints)
SATJOB coded: 0-3 (4 categories, 3 cutpoints)
SATFIN coded: 0-2 (3 categories, 2 cutpoints)


In [8]:
# Prepare predictors

# Binary encodings
df_analysis['female'] = (df_analysis['sex'] == 2).astype(int)
df_analysis['fulltime'] = (df_analysis['wrkstat'] == 1).astype(int)  # Full-time vs other
df_analysis['parttime'] = (df_analysis['wrkstat'] == 2).astype(int)  # Part-time indicator

# Race dummies (reference: White=1)
df_analysis['race_black'] = (df_analysis['race'] == 2).astype(int)
df_analysis['race_other'] = (df_analysis['race'] == 3).astype(int)

# Religion: simplify to religious affiliation vs none
# relig=4 is 'None' in GSS coding
df_analysis['religious'] = (df_analysis['relig'] != 4).astype(int)

# Standardize continuous/ordinal predictors
predictors_to_std = ['hlthdep', 'stress', 'wrkmeangfl', 'finrela', 'anxiety', 'age', 'degree',
                     'hours_worked', 'chngtme']
binary_predictors = ['female', 'fulltime', 'race_black', 'race_other', 'religious']
predictor_names = predictors_to_std + binary_predictors

# Store standardization parameters
std_params = {}
for var in predictors_to_std:
    mean_val = df_analysis[var].mean()
    std_val = df_analysis[var].std()
    df_analysis[f'{var}_std'] = (df_analysis[var] - mean_val) / std_val
    std_params[var] = {'mean': mean_val, 'std': std_val}

# Build design matrix
X_cols = [f'{v}_std' for v in predictors_to_std] + binary_predictors
X = df_analysis[X_cols].values

print(f'Design matrix shape: {X.shape}')
print(f'Predictors: {predictor_names}')

Design matrix shape: (479, 14)
Predictors: ['hlthdep', 'stress', 'wrkmeangfl', 'finrela', 'anxiety', 'age', 'degree', 'hours_worked', 'chngtme', 'female', 'fulltime', 'race_black', 'race_other', 'religious']


In [9]:
# Predictor correlation matrix
corr_matrix = df_analysis[X_cols].corr()
corr_matrix.columns = predictor_names
corr_matrix.index = predictor_names

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(corr_matrix.values, 2),
    texttemplate='%{text}'
))
fig.update_layout(title='Predictor Correlations', width=700, height=600)
fig.show()

## Bivariate Normal CDF Implementation

The key computational challenge is evaluating the bivariate normal CDF for rectangle probabilities. We need:

$$
P(Y_1=k, Y_2=j) = \Phi_2(\theta_{1,k}, \theta_{2,j}; \rho) - \Phi_2(\theta_{1,k-1}, \theta_{2,j}; \rho) - \Phi_2(\theta_{1,k}, \theta_{2,j-1}; \rho) + \Phi_2(\theta_{1,k-1}, \theta_{2,j-1}; \rho)
$$

We'll use an approximation based on the Drezner-Wesolowsky method.

In [10]:
def bvn_cdf_approx(x, y, rho):
    """
    Approximate bivariate normal CDF using Drezner-Wesolowsky (1990) method.
    
    This computes P(X <= x, Y <= y) where (X, Y) ~ BVN(0, 0, 1, 1, rho).
    Uses Gauss-Legendre quadrature for numerical integration.
    """
    # Gauss-Legendre weights and abscissas for 6-point quadrature
    w = np.array([0.1713244923791705, 0.3607615730481384, 0.4679139345726904,
                  0.4679139345726904, 0.3607615730481384, 0.1713244923791705])
    xi = np.array([-0.9324695142031522, -0.6612093864662647, -0.2386191860831970,
                   0.2386191860831970, 0.6612093864662647, 0.9324695142031522])
    
    # Handle edge cases
    if np.abs(rho) < 1e-10:
        return stats.norm.cdf(x) * stats.norm.cdf(y)
    
    if rho > 0.9999:
        return stats.norm.cdf(min(x, y))
    
    if rho < -0.9999:
        return max(0, stats.norm.cdf(x) - stats.norm.cdf(-y))
    
    # Drezner-Wesolowsky approximation
    h = -x
    k = -y
    hk = h * k
    
    if np.abs(rho) < 0.925:
        # Use direct formula for moderate correlations
        hs = (h * h + k * k) / 2
        asr = np.arcsin(rho)
        sn = np.sin(asr * (1 + xi) / 2)
        bvn = np.sum(w * np.exp((sn * hk - hs) / (1 - sn * sn)))
        bvn = bvn * asr / (4 * np.pi) + stats.norm.cdf(-h) * stats.norm.cdf(-k)
    else:
        # Use alternative formula for high correlations
        if rho < 0:
            k = -k
            hk = -hk
        
        if np.abs(rho) < 1:
            ass = (1 - rho) * (1 + rho)
            a = np.sqrt(ass)
            bs = (h - k) ** 2
            c = (4 - hk) / 8
            d = (12 - hk) / 16
            asr = -(bs / ass + hk) / 2
            if asr > -100:
                bvn = a * np.exp(asr) * (1 - c * (bs - ass) * (1 - d * bs / 5) / 3 + c * d * ass * ass / 5)
            else:
                bvn = 0
            
            if -hk < 100:
                b = np.sqrt(bs)
                bvn = bvn - np.exp(-hk / 2) * np.sqrt(2 * np.pi) * stats.norm.cdf(-b / a) * b * (1 - c * bs * (1 - d * bs / 5) / 3)
            
            a = a / 2
            xs = (a * (1 + xi)) ** 2
            asr = -(bs / xs + hk) / 2
            valid = asr > -100
            if np.any(valid):
                asr_valid = np.where(valid, asr, -100)
                bvn = bvn + a * np.sum(w * np.exp(asr_valid) * (np.exp(-hk * (1 - xs) / (2 * (1 + np.sqrt(1 - xs)))) / np.sqrt(1 - xs) - (1 + c * xs * (1 + d * xs))))
            
            bvn = -bvn / (2 * np.pi)
        
        if rho > 0:
            bvn = bvn + stats.norm.cdf(-max(h, k))
        else:
            bvn = -bvn
            if k > h:
                bvn = bvn + stats.norm.cdf(k) - stats.norm.cdf(h)
    
    return max(0, min(1, bvn))

# Vectorized version for arrays
bvn_cdf_vec = np.vectorize(bvn_cdf_approx)

# Test the implementation
print('Testing BVN CDF approximation:')
print(f'  P(X<0, Y<0 | rho=0.5): {bvn_cdf_approx(0, 0, 0.5):.4f} (expected ~0.333)')
print(f'  P(X<0, Y<0 | rho=0.0): {bvn_cdf_approx(0, 0, 0.0):.4f} (expected 0.250)')
print(f'  P(X<0, Y<0 | rho=-0.5): {bvn_cdf_approx(0, 0, -0.5):.4f} (expected ~0.167)')

Testing BVN CDF approximation:
  P(X<0, Y<0 | rho=0.5): 0.3333 (expected ~0.333)
  P(X<0, Y<0 | rho=0.0): 0.2500 (expected 0.250)
  P(X<0, Y<0 | rho=-0.5): 0.1667 (expected ~0.167)


## PyTensor Implementation of Bivariate Normal CDF

We need a differentiable version for use in PyMC. We'll use the Owen's T function approach which has a simpler form for automatic differentiation.

In [11]:
def owens_t_approx(h, a):
    """
    Owen's T function approximation using series expansion.
    T(h, a) = (1/2π) ∫₀ᵃ exp(-h²(1+t²)/2) / (1+t²) dt
    """
    # Use Gaussian quadrature
    n_points = 10
    t, w = np.polynomial.legendre.leggauss(n_points)
    
    # Transform from [-1, 1] to [0, a]
    t_scaled = a * (t + 1) / 2
    w_scaled = w * a / 2
    
    integrand = np.exp(-h**2 * (1 + t_scaled**2) / 2) / (1 + t_scaled**2)
    return np.sum(w_scaled * integrand) / (2 * np.pi)

def bvn_cdf_owens(x, y, rho):
    """
    Bivariate normal CDF using Owen's T function.
    P(X <= x, Y <= y) where (X, Y) ~ BVN(0, 0, 1, 1, rho)
    """
    if np.abs(rho) < 1e-10:
        return stats.norm.cdf(x) * stats.norm.cdf(y)
    
    # Formula: Φ₂(x, y; ρ) = Φ(x)Φ(y) + T(x, (y-ρx)/√(1-ρ²)/x) + T(y, (x-ρy)/√(1-ρ²)/y)
    # when x, y > 0
    
    # Use the Drezner formula which is more stable
    return bvn_cdf_approx(x, y, rho)

# For PyTensor, we'll create an Op that wraps scipy's multivariate normal
from pytensor.tensor import as_tensor_variable
from pytensor.graph.op import Op
from pytensor.graph.basic import Apply

class BivariateNormalCDF(Op):
    """
    PyTensor Op for bivariate normal CDF.
    """
    __props__ = ()
    
    def make_node(self, x, y, rho):
        x = as_tensor_variable(x)
        y = as_tensor_variable(y)
        rho = as_tensor_variable(rho)
        return Apply(self, [x, y, rho], [x.type()])
    
    def perform(self, node, inputs, output_storage):
        x, y, rho = inputs
        # Use scipy's mvn for accurate computation
        result = np.zeros_like(x)
        for i in range(len(x)):
            result[i] = bvn_cdf_approx(x[i], y[i], rho)
        output_storage[0][0] = result
    
    def grad(self, inputs, output_grads):
        x, y, rho = inputs
        gz = output_grads[0]
        
        # Gradients of BVN CDF
        # ∂Φ₂/∂x = φ(x) Φ((y - ρx) / √(1-ρ²))
        # ∂Φ₂/∂y = φ(y) Φ((x - ρy) / √(1-ρ²))
        
        sqrt_1_rho2 = pt.sqrt(1 - rho**2)
        
        phi_x = pt.exp(-x**2 / 2) / pt.sqrt(2 * np.pi)
        phi_y = pt.exp(-y**2 / 2) / pt.sqrt(2 * np.pi)
        
        Phi_cond_x = 0.5 * (1 + pt.erf((y - rho * x) / (sqrt_1_rho2 * pt.sqrt(2))))
        Phi_cond_y = 0.5 * (1 + pt.erf((x - rho * y) / (sqrt_1_rho2 * pt.sqrt(2))))
        
        grad_x = gz * phi_x * Phi_cond_x
        grad_y = gz * phi_y * Phi_cond_y
        
        # Gradient w.r.t. rho is more complex - use finite differences or derive
        # For now, use a numerical approximation
        grad_rho = pt.zeros_like(rho)  # Simplified - could be improved
        
        return [grad_x, grad_y, grad_rho]

bvn_cdf_op = BivariateNormalCDF()

print('BivariateNormalCDF Op created successfully')

BivariateNormalCDF Op created successfully


## Alternative Approach: Marginalized Likelihood with Numerical Integration

Given the complexity of implementing a fully differentiable multivariate normal CDF, we'll use an alternative approach:

1. **First**, fit marginal ordinal probit models for each outcome separately
2. **Then**, estimate the polychoric correlations from the latent residuals
3. **Finally**, fit a combined model using the estimated correlations as informative constraints

This two-stage approach is computationally simpler and provides good estimates.

In [12]:
# Extract data for modeling
y1 = df_analysis['y1'].values  # LIFENOW (0-9)
y2 = df_analysis['y2'].values  # SATJOB (0-3)
y3 = df_analysis['y3'].values  # SATFIN (0-2)
N = len(y1)

print(f'Sample size: {N}')
print(f'LIFENOW categories: {K1} (0 to {K1-1})')
print(f'SATJOB categories: {K2} (0 to {K2-1})')
print(f'SATFIN categories: {K3} (0 to {K3-1})')

Sample size: 479
LIFENOW categories: 10 (0 to 9)
SATJOB categories: 4 (0 to 3)
SATFIN categories: 3 (0 to 2)


## Model 1: Marginal Ordinal Probit for LIFENOW

In [13]:
# Horseshoe prior hyperparameters
D = len(predictor_names)  # Number of predictors
D0 = 4  # Expected number of important predictors (prior sparsity belief)

coords_lifenow = {
    'predictors': predictor_names,
    'obs': np.arange(N)
}

with pm.Model(coords=coords_lifenow) as model_lifenow:
    # Data
    X_data = pm.Data('X', X)
    y_data = pm.Data('y', y1)
    
    # Regularized horseshoe prior on regression coefficients (Piironen & Vehtari 2017)
    beta = horseshoe_prior('beta', D=D, N=N, D0=D0, dims='predictors')
    
    # Linear predictor
    eta = pm.math.dot(X_data, beta)
    
    # Ordered cutpoints for probit model (9 cutpoints for 10 categories)
    # Use cumulative sum of positive increments to ensure ordering
    cutpoint_deltas = pm.Exponential('cutpoint_deltas', lam=1, shape=K1-2)
    cutpoint_base = pm.Normal('cutpoint_base', mu=0, sigma=2)
    cutpoints = pm.Deterministic(
        'cutpoints',
        pt.concatenate([[cutpoint_base], cutpoint_base + pt.cumsum(cutpoint_deltas)])
    )
    
    # Likelihood using OrderedProbit
    y_obs = pm.OrderedProbit('y_obs', eta=eta, cutpoints=cutpoints, observed=y_data)

print('LIFENOW model structure:')
print(model_lifenow)

LIFENOW model structure:


In [14]:
# Fit LIFENOW model
with model_lifenow:
    trace_lifenow = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        nuts_sampler='nutpie',
        random_seed=42
    )


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,1,0.14,223
,2000,10,0.13,63
,2000,6,0.12,63
,2000,10,0.14,31


In [15]:
# Check convergence
print('=== LIFENOW Model Diagnostics ===')
if 'diverging' in trace_lifenow.sample_stats:
    n_div = int(trace_lifenow.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

summary_lifenow = az.summary(trace_lifenow, var_names=['beta'])
print('\nCoefficient estimates (LIFENOW):')
print(summary_lifenow[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

=== LIFENOW Model Diagnostics ===
Divergent transitions: 27

Coefficient estimates (LIFENOW):
                     mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
beta[hlthdep]      -0.403  0.060  -0.521   -0.295    1.0    4305.0
beta[stress]        0.085  0.052  -0.006    0.180    1.0    3039.0
beta[wrkmeangfl]    0.233  0.052   0.130    0.325    1.0    4670.0
beta[finrela]       0.235  0.057   0.123    0.338    1.0    4929.0
beta[anxiety]      -0.029  0.047  -0.129    0.049    1.0    3586.0
beta[age]           0.030  0.041  -0.041    0.113    1.0    3576.0
beta[degree]        0.083  0.055  -0.009    0.185    1.0    2934.0
beta[hours_worked] -0.003  0.036  -0.081    0.062    1.0    5104.0
beta[chngtme]       0.031  0.041  -0.039    0.112    1.0    3407.0
beta[female]       -0.001  0.059  -0.129    0.116    1.0    4983.0
beta[fulltime]     -0.010  0.072  -0.165    0.130    1.0    4608.0
beta[race_black]   -0.061  0.096  -0.272    0.082    1.0    3997.0
beta[race_other]   -0.069  0.103  -

## Model 2: Marginal Ordinal Probit for SATJOB

In [16]:
coords_satjob = {
    'predictors': predictor_names,
    'obs': np.arange(N)
}

with pm.Model(coords=coords_satjob) as model_satjob:
    # Data
    X_data = pm.Data('X', X)
    y_data = pm.Data('y', y2)
    
    # Regularized horseshoe prior on regression coefficients (Piironen & Vehtari 2017)
    beta = horseshoe_prior('beta', D=D, N=N, D0=D0, dims='predictors')
    
    # Linear predictor
    eta = pm.math.dot(X_data, beta)
    
    # Ordered cutpoints for probit model (3 cutpoints for 4 categories)
    cutpoint_deltas = pm.Exponential('cutpoint_deltas', lam=1, shape=K2-2)
    cutpoint_base = pm.Normal('cutpoint_base', mu=0, sigma=2)
    cutpoints = pm.Deterministic(
        'cutpoints',
        pt.concatenate([[cutpoint_base], cutpoint_base + pt.cumsum(cutpoint_deltas)])
    )
    
    # Likelihood using OrderedProbit
    y_obs = pm.OrderedProbit('y_obs', eta=eta, cutpoints=cutpoints, observed=y_data)

print('SATJOB model structure:')
print(model_satjob)

SATJOB model structure:


In [17]:
# Fit SATJOB model
with model_satjob:
    trace_satjob = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        nuts_sampler='nutpie',
        random_seed=42
    )

Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,1,0.19,63
,2000,0,0.18,31
,2000,1,0.17,31
,2000,1,0.17,31


In [18]:
# Check convergence
print('=== SATJOB Model Diagnostics ===')
if 'diverging' in trace_satjob.sample_stats:
    n_div = int(trace_satjob.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

summary_satjob = az.summary(trace_satjob, var_names=['beta'])
print('\nCoefficient estimates (SATJOB):')
print(summary_satjob[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

=== SATJOB Model Diagnostics ===
Divergent transitions: 3

Coefficient estimates (SATJOB):
                     mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
beta[hlthdep]      -0.158  0.068  -0.286   -0.032    1.0    3180.0
beta[stress]        0.278  0.062   0.158    0.394    1.0    3608.0
beta[wrkmeangfl]    0.504  0.061   0.383    0.616    1.0    4425.0
beta[finrela]       0.021  0.046  -0.067    0.110    1.0    4118.0
beta[anxiety]       0.006  0.051  -0.093    0.108    1.0    4525.0
beta[age]           0.146  0.061   0.021    0.256    1.0    2544.0
beta[degree]       -0.029  0.047  -0.121    0.056    1.0    3525.0
beta[hours_worked]  0.039  0.055  -0.053    0.147    1.0    2653.0
beta[chngtme]       0.148  0.058   0.030    0.252    1.0    2844.0
beta[female]       -0.177  0.114  -0.382    0.018    1.0    2821.0
beta[fulltime]      0.224  0.155  -0.032    0.500    1.0    1697.0
beta[race_black]   -0.031  0.093  -0.228    0.140    1.0    5510.0
beta[race_other]   -0.038  0.102  -0.2

## Model 3: Marginal Ordinal Probit for SATFIN

In [19]:
coords_satfin = {
    'predictors': predictor_names,
    'obs': np.arange(N)
}

with pm.Model(coords=coords_satfin) as model_satfin:
    # Data
    X_data = pm.Data('X', X)
    y_data = pm.Data('y', y3)
    
    # Regularized horseshoe prior on regression coefficients (Piironen & Vehtari 2017)
    beta = horseshoe_prior('beta', D=D, N=N, D0=D0, dims='predictors')
    
    # Linear predictor
    eta = pm.math.dot(X_data, beta)
    
    # Ordered cutpoints for probit model (2 cutpoints for 3 categories)
    cutpoint_deltas = pm.Exponential('cutpoint_deltas', lam=1, shape=K3-2)
    cutpoint_base = pm.Normal('cutpoint_base', mu=0, sigma=2)
    cutpoints = pm.Deterministic(
        'cutpoints',
        pt.concatenate([[cutpoint_base], cutpoint_base + pt.cumsum(cutpoint_deltas)])
    )
    
    # Likelihood using OrderedProbit
    y_obs = pm.OrderedProbit('y_obs', eta=eta, cutpoints=cutpoints, observed=y_data)

print('SATFIN model structure:')
print(model_satfin)

SATFIN model structure:


In [20]:
# Fit SATFIN model
with model_satfin:
    trace_satfin = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.9,
        nuts_sampler='nutpie',
        random_seed=42
    )


Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.21,31
,2000,0,0.18,31
,2000,0,0.21,15
,2000,0,0.19,31


In [21]:
# Check convergence
print('=== SATFIN Model Diagnostics ===')
if 'diverging' in trace_satfin.sample_stats:
    n_div = int(trace_satfin.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

summary_satfin = az.summary(trace_satfin, var_names=['beta'])
print('\nCoefficient estimates (SATFIN):')
print(summary_satfin[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk']])

=== SATFIN Model Diagnostics ===
Divergent transitions: 0

Coefficient estimates (SATFIN):
                     mean     sd  hdi_3%  hdi_97%  r_hat  ess_bulk
beta[hlthdep]      -0.246  0.068  -0.367   -0.110    1.0    4750.0
beta[stress]        0.029  0.046  -0.048    0.125    1.0    3725.0
beta[wrkmeangfl]    0.042  0.049  -0.036    0.140    1.0    3670.0
beta[finrela]       0.478  0.065   0.359    0.600    1.0    4827.0
beta[anxiety]      -0.032  0.052  -0.147    0.048    1.0    3838.0
beta[age]           0.030  0.046  -0.045    0.125    1.0    3744.0
beta[degree]        0.180  0.063   0.061    0.297    1.0    4484.0
beta[hours_worked] -0.016  0.041  -0.103    0.058    1.0    4475.0
beta[chngtme]       0.080  0.054  -0.012    0.180    1.0    2822.0
beta[female]       -0.083  0.091  -0.270    0.050    1.0    3176.0
beta[fulltime]      0.003  0.076  -0.141    0.167    1.0    3553.0
beta[race_black]   -0.061  0.102  -0.300    0.082    1.0    4118.0
beta[race_other]    0.004  0.081  -0.1

## Coefficient Comparison: LIFENOW, SATJOB, and SATFIN

In [22]:
# Extract posterior means and HDIs for all three outcomes
beta_lifenow = trace_lifenow.posterior['beta'].values.reshape(-1, len(predictor_names))
beta_satjob = trace_satjob.posterior['beta'].values.reshape(-1, len(predictor_names))
beta_satfin = trace_satfin.posterior['beta'].values.reshape(-1, len(predictor_names))

comparison_df = pd.DataFrame({
    'Predictor': predictor_names,
    'LIFENOW_mean': beta_lifenow.mean(axis=0),
    'LIFENOW_hdi_low': np.percentile(beta_lifenow, 3, axis=0),
    'LIFENOW_hdi_high': np.percentile(beta_lifenow, 97, axis=0),
    'SATJOB_mean': beta_satjob.mean(axis=0),
    'SATJOB_hdi_low': np.percentile(beta_satjob, 3, axis=0),
    'SATJOB_hdi_high': np.percentile(beta_satjob, 97, axis=0),
    'SATFIN_mean': beta_satfin.mean(axis=0),
    'SATFIN_hdi_low': np.percentile(beta_satfin, 3, axis=0),
    'SATFIN_hdi_high': np.percentile(beta_satfin, 97, axis=0),
})

print('Coefficient Comparison (standardized predictors):')
print('Note: All outcomes coded so higher = more satisfied')
print(comparison_df[['Predictor', 'LIFENOW_mean', 'SATJOB_mean', 'SATFIN_mean']].round(3))

Coefficient Comparison (standardized predictors):
Note: All outcomes coded so higher = more satisfied
       Predictor  LIFENOW_mean  SATJOB_mean  SATFIN_mean
0        hlthdep        -0.403       -0.158       -0.246
1         stress         0.085        0.278        0.029
2     wrkmeangfl         0.233        0.504        0.042
3        finrela         0.235        0.021        0.478
4        anxiety        -0.029        0.006       -0.032
5            age         0.030        0.146        0.030
6         degree         0.083       -0.029        0.180
7   hours_worked        -0.003        0.039       -0.016
8        chngtme         0.031        0.148        0.080
9         female        -0.001       -0.177       -0.083
10      fulltime        -0.010        0.224        0.003
11    race_black        -0.061       -0.031       -0.061
12    race_other        -0.069       -0.038        0.004
13     religious         0.054       -0.006       -0.003


## Shrinkage Analysis

The regularized horseshoe prior uses a **non-centered parameterization**:

$$\beta_j = z_j \cdot \tau \cdot \tilde{\lambda}_j$$

where $z_j \sim \mathcal{N}(0, 1)$, $\tau$ is the global shrinkage, and $\tilde{\lambda}_j$ is the regularized local shrinkage.

**Key insight**: In the non-centered parameterization, shrinkage works differently than the classic formula suggests. When data strongly supports a non-zero coefficient, the signal is captured in $z_j$ being pushed away from zero, while $\tau$ remains small globally.

**Diagnostics**:
- $|z_j| > 2$: Strong signal (coefficient escapes shrinkage)
- $|z_j| \approx 1$: Moderate signal
- $|z_j| < 0.5$: Weak signal (coefficient shrunk toward zero)
- HDI excluding zero: Standard Bayesian significance check

In [23]:
# Shrinkage Analysis for Regularized Horseshoe (Non-Centered Parameterization)
#
# IMPORTANT: The classic shrinkage factor kappa = 1/(1 + tau^2 * lambda^2) assumes
# a centered parameterization. In the non-centered parameterization used here:
#   z ~ N(0, 1)
#   beta = z * tau * lam_tilde
#
# The signal is captured in z, not in (tau, lambda). When data supports a non-zero
# coefficient, z gets pushed away from 0, while tau stays small globally.
#
# Better diagnostics for non-centered horseshoe:
# 1. |z| magnitude: |z| > 2 suggests strong signal
# 2. P(|beta| > threshold): probability coefficient is practically significant
# 3. HDI excluding zero: standard credible interval check

def compute_shrinkage_diagnostics(trace, name='beta', threshold=0.05):
    """Compute shrinkage diagnostics for non-centered horseshoe.
    
    Returns:
    - z_magnitude: Mean |z| for each predictor (>2 suggests signal)
    - signal_prob: P(|beta| > threshold) for each predictor
    - hdi_excludes_zero: Whether 94% HDI excludes zero
    """
    z_samples = trace.posterior[f'{name}_z'].values
    beta_samples = trace.posterior[name].values
    
    # Reshape to (samples, predictors)
    z_flat = z_samples.reshape(-1, z_samples.shape[-1])
    beta_flat = beta_samples.reshape(-1, beta_samples.shape[-1])
    
    # |z| magnitude (signal strength in non-centered parameterization)
    z_magnitude = np.abs(z_flat).mean(axis=0)
    
    # P(|beta| > threshold)
    signal_prob = (np.abs(beta_flat) > threshold).mean(axis=0)
    
    # 94% HDI excludes zero
    hdi_lower = np.percentile(beta_flat, 3, axis=0)
    hdi_upper = np.percentile(beta_flat, 97, axis=0)
    hdi_excludes_zero = (hdi_lower > 0) | (hdi_upper < 0)
    
    # Beta posterior mean and SD
    beta_mean = beta_flat.mean(axis=0)
    beta_sd = beta_flat.std(axis=0)
    
    return {
        'z_magnitude': z_magnitude,
        'signal_prob': signal_prob,
        'hdi_excludes_zero': hdi_excludes_zero,
        'beta_mean': beta_mean,
        'beta_sd': beta_sd,
        'hdi_lower': hdi_lower,
        'hdi_upper': hdi_upper
    }

# Compute for each model
diag_lifenow = compute_shrinkage_diagnostics(trace_lifenow)
diag_satjob = compute_shrinkage_diagnostics(trace_satjob)
diag_satfin = compute_shrinkage_diagnostics(trace_satfin)

# Create summary dataframe
shrinkage_summary = pd.DataFrame({
    'Predictor': predictor_names,
    'LIFENOW_|z|': diag_lifenow['z_magnitude'],
    'LIFENOW_beta': diag_lifenow['beta_mean'],
    'LIFENOW_signal': diag_lifenow['hdi_excludes_zero'],
    'SATJOB_|z|': diag_satjob['z_magnitude'],
    'SATJOB_beta': diag_satjob['beta_mean'],
    'SATJOB_signal': diag_satjob['hdi_excludes_zero'],
    'SATFIN_|z|': diag_satfin['z_magnitude'],
    'SATFIN_beta': diag_satfin['beta_mean'],
    'SATFIN_signal': diag_satfin['hdi_excludes_zero'],
})

print('=== SHRINKAGE DIAGNOSTICS (Non-Centered Horseshoe) ===')
print()
print('Interpretation:')
print('  |z| > 2: Strong signal (coefficient escapes shrinkage)')
print('  |z| ~ 1: Moderate signal')
print('  |z| < 0.5: Weak signal (coefficient shrunk toward zero)')
print('  signal: True if 94% HDI excludes zero')
print()
print(shrinkage_summary.to_string(index=False, float_format='{:.3f}'.format))

# Count signals
print(f'\n=== Signal Summary ===')
print(f'LIFENOW: {diag_lifenow["hdi_excludes_zero"].sum()}/{len(predictor_names)} predictors with HDI excluding zero')
print(f'SATJOB:  {diag_satjob["hdi_excludes_zero"].sum()}/{len(predictor_names)} predictors with HDI excluding zero')
print(f'SATFIN:  {diag_satfin["hdi_excludes_zero"].sum()}/{len(predictor_names)} predictors with HDI excluding zero')

=== SHRINKAGE DIAGNOSTICS (Non-Centered Horseshoe) ===

Interpretation:
  |z| > 2: Strong signal (coefficient escapes shrinkage)
  |z| ~ 1: Moderate signal
  |z| < 0.5: Weak signal (coefficient shrunk toward zero)
  signal: True if 94% HDI excludes zero

   Predictor  LIFENOW_|z|  LIFENOW_beta  LIFENOW_signal  SATJOB_|z|  SATJOB_beta  SATJOB_signal  SATFIN_|z|  SATFIN_beta  SATFIN_signal
     hlthdep        1.618        -0.403            True       1.011       -0.158           True       1.364       -0.246           True
      stress        0.873         0.085           False       1.278        0.278           True       0.633        0.029          False
  wrkmeangfl        1.334         0.233            True       1.576        0.504           True       0.679        0.042          False
     finrela        1.338         0.235            True       0.574        0.021          False       1.710        0.478           True
     anxiety        0.622        -0.029           False       0.5

In [24]:
# Visualize signal strength (|z| magnitude) across models
fig = make_subplots(rows=1, cols=3, subplot_titles=['LIFENOW', 'SATJOB', 'SATFIN'],
                    shared_yaxes=True, horizontal_spacing=0.05)

for col, (diag, outcome) in enumerate([(diag_lifenow, 'LIFENOW'), 
                                        (diag_satjob, 'SATJOB'), 
                                        (diag_satfin, 'SATFIN')], 1):
    z_mag = diag['z_magnitude']
    signal = diag['hdi_excludes_zero']
    # Sort by signal strength (strongest to weakest)
    sort_idx = np.argsort(z_mag)[::-1]
    
    for i, idx in enumerate(sort_idx):
        # Color: green for signal (high |z|), red for noise (low |z|)
        # Normalize z_magnitude to 0-1 range (cap at 3 for visualization)
        z_norm = min(z_mag[idx] / 3, 1)
        color = f'rgb({int((1-z_norm)*255)}, {int(z_norm*200)}, 0)'
        
        # Add star marker if HDI excludes zero
        name = predictor_names[idx] + (' *' if signal[idx] else '')
        
        fig.add_trace(
            go.Bar(x=[z_mag[idx]], y=[name], 
                   orientation='h', marker_color=color,
                   showlegend=False, hovertemplate=f'{predictor_names[idx]}: |z|=%{{x:.2f}}'),
            row=1, col=col
        )

fig.update_xaxes(range=[0, 3], title_text='Mean |z| (signal strength)', row=1, col=2)
fig.update_layout(
    title='Signal Strength by Predictor and Outcome<br><sub>High |z| (green) = signal preserved, Low |z| (red) = shrunk toward zero. * = 94% HDI excludes zero</sub>',
    height=500, width=1000,
    showlegend=False
)
fig.show()

# List predictors with signal (HDI excludes zero) in at least one outcome
signal_predictors = [p for j, p in enumerate(predictor_names) 
                     if diag_lifenow['hdi_excludes_zero'][j] or 
                        diag_satjob['hdi_excludes_zero'][j] or 
                        diag_satfin['hdi_excludes_zero'][j]]
print(f'\nPredictors with 94% HDI excluding zero in at least one outcome: {signal_predictors}')


Predictors with 94% HDI excluding zero in at least one outcome: ['hlthdep', 'stress', 'wrkmeangfl', 'finrela', 'age', 'degree', 'chngtme']


In [25]:
# Visualize coefficient comparison for all three outcomes
fig = make_subplots(rows=1, cols=3, subplot_titles=['LIFENOW', 'SATJOB', 'SATFIN'], shared_yaxes=True)

colors = ['blue', 'red', 'green']
outcome_data = [
    (comparison_df['LIFENOW_mean'], comparison_df['LIFENOW_hdi_low'], comparison_df['LIFENOW_hdi_high']),
    (comparison_df['SATJOB_mean'], comparison_df['SATJOB_hdi_low'], comparison_df['SATJOB_hdi_high']),
    (comparison_df['SATFIN_mean'], comparison_df['SATFIN_hdi_low'], comparison_df['SATFIN_hdi_high']),
]

for col, (means, hdi_low, hdi_high) in enumerate(outcome_data, 1):
    fig.add_trace(go.Scatter(
        x=means,
        y=comparison_df['Predictor'],
        mode='markers',
        marker=dict(size=10, color=colors[col-1]),
        error_x=dict(
            type='data',
            symmetric=False,
            array=hdi_high - means,
            arrayminus=means - hdi_low
        ),
        showlegend=False
    ), row=1, col=col)
    fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=col)

fig.update_layout(
    title='Marginal Model Coefficients by Outcome (94% HDI)',
    height=450, width=1100
)
for col in range(1, 4):
    fig.update_xaxes(title_text='Coefficient', row=1, col=col)
fig.show()

## Polychoric Correlation Estimation

Now we estimate the correlations between the latent variables after accounting for predictors. We compute "residuals" on the latent scale and estimate their 3×3 correlation matrix.

In [26]:
# Get posterior means for predictions from all three marginal models
beta1_mean = beta_lifenow.mean(axis=0)
beta2_mean = beta_satjob.mean(axis=0)
beta3_mean = beta_satfin.mean(axis=0)

cutpoints1_mean = trace_lifenow.posterior['cutpoints'].values.reshape(-1, K1-1).mean(axis=0)
cutpoints2_mean = trace_satjob.posterior['cutpoints'].values.reshape(-1, K2-1).mean(axis=0)
cutpoints3_mean = trace_satfin.posterior['cutpoints'].values.reshape(-1, K3-1).mean(axis=0)

# Linear predictors
eta1 = X @ beta1_mean
eta2 = X @ beta2_mean
eta3 = X @ beta3_mean

print(f'Linear predictor ranges:')
print(f'  LIFENOW eta: [{eta1.min():.2f}, {eta1.max():.2f}]')
print(f'  SATJOB eta: [{eta2.min():.2f}, {eta2.max():.2f}]')
print(f'  SATFIN eta: [{eta3.min():.2f}, {eta3.max():.2f}]')
print(f'\nCutpoints:')
print(f'  LIFENOW: {cutpoints1_mean.round(2)}')
print(f'  SATJOB: {cutpoints2_mean.round(2)}')
print(f'  SATFIN: {cutpoints3_mean.round(2)}')

Linear predictor ranges:
  LIFENOW eta: [-2.16, 1.49]
  SATJOB eta: [-2.77, 1.94]
  SATFIN eta: [-2.00, 1.76]

Cutpoints:
  LIFENOW: [-4.49 -4.03 -3.01 -2.44 -1.68 -1.08 -0.68  0.18  1.39]
  SATJOB: [-2.31 -1.36  0.28]
  SATFIN: [-0.73  1.05]


In [27]:
def compute_latent_residuals(y_obs, eta, cutpoints):
    """
    Compute expected latent residuals for ordinal probit model.
    
    For observation i with y_i = k:
    E[z_i | y_i = k] = E[z_i | θ_{k-1} < z_i - η_i ≤ θ_k]
    
    This is the mean of a truncated normal.
    """
    n = len(y_obs)
    K = len(cutpoints) + 1
    
    # Extended cutpoints with -inf and inf
    cutpoints_ext = np.concatenate([[-np.inf], cutpoints, [np.inf]])
    
    residuals = np.zeros(n)
    
    for i in range(n):
        k = int(y_obs[i])
        # Bounds for truncated normal (on residual scale)
        a = cutpoints_ext[k] - eta[i]
        b = cutpoints_ext[k + 1] - eta[i]
        
        # Mean of truncated standard normal on (a, b)
        if np.isinf(a) and a < 0:
            # Left tail
            residuals[i] = -stats.norm.pdf(b) / stats.norm.cdf(b)
        elif np.isinf(b) and b > 0:
            # Right tail
            residuals[i] = stats.norm.pdf(a) / (1 - stats.norm.cdf(a))
        else:
            # Interior
            alpha = stats.norm.cdf(a)
            beta = stats.norm.cdf(b)
            if beta - alpha > 1e-10:
                residuals[i] = (stats.norm.pdf(a) - stats.norm.pdf(b)) / (beta - alpha)
            else:
                residuals[i] = (a + b) / 2  # Approximate for very narrow intervals
    
    return residuals

# Compute latent residuals
resid1 = compute_latent_residuals(y1, eta1, cutpoints1_mean)
resid2 = compute_latent_residuals(y2, eta2, cutpoints2_mean)

print(f'Latent residual statistics:')
print(f'  LIFENOW: mean={resid1.mean():.3f}, std={resid1.std():.3f}')
print(f'  SATJOB: mean={resid2.mean():.3f}, std={resid2.std():.3f}')

Latent residual statistics:
  LIFENOW: mean=0.001, std=0.938
  SATJOB: mean=-0.002, std=0.829


In [28]:
# Compute latent residuals for all three outcomes
resid1 = compute_latent_residuals(y1, eta1, cutpoints1_mean)
resid2 = compute_latent_residuals(y2, eta2, cutpoints2_mean)
resid3 = compute_latent_residuals(y3, eta3, cutpoints3_mean)

print(f'Latent residual statistics:')
print(f'  LIFENOW: mean={resid1.mean():.3f}, std={resid1.std():.3f}')
print(f'  SATJOB: mean={resid2.mean():.3f}, std={resid2.std():.3f}')
print(f'  SATFIN: mean={resid3.mean():.3f}, std={resid3.std():.3f}')

# Compute 3x3 polychoric correlation matrix
residuals = np.column_stack([resid1, resid2, resid3])
polychoric_corr_matrix = np.corrcoef(residuals.T)

print(f'\n=== POLYCHORIC CORRELATION MATRIX (Residuals) ===')
outcome_labels = ['LIFENOW', 'SATJOB', 'SATFIN']
polychoric_df = pd.DataFrame(polychoric_corr_matrix, 
                              index=outcome_labels, 
                              columns=outcome_labels)
print(polychoric_df.round(3))
print('\nNote: These are residual correlations AFTER accounting for predictors.')

Latent residual statistics:
  LIFENOW: mean=0.001, std=0.938
  SATJOB: mean=-0.002, std=0.829
  SATFIN: mean=0.001, std=0.837

=== POLYCHORIC CORRELATION MATRIX (Residuals) ===
         LIFENOW  SATJOB  SATFIN
LIFENOW    1.000   0.065   0.221
SATJOB     0.065   1.000   0.090
SATFIN     0.221   0.090   1.000

Note: These are residual correlations AFTER accounting for predictors.


In [29]:
# Visualize latent residuals - pairwise scatter plots
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    f'LIFENOW vs SATJOB (r={polychoric_corr_matrix[0,1]:.3f})',
    f'LIFENOW vs SATFIN (r={polychoric_corr_matrix[0,2]:.3f})',
    f'SATJOB vs SATFIN (r={polychoric_corr_matrix[1,2]:.3f})'
])

fig.add_trace(go.Scatter(x=resid1, y=resid2, mode='markers', marker=dict(opacity=0.4, size=5),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=resid1, y=resid3, mode='markers', marker=dict(opacity=0.4, size=5),
                         showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=resid2, y=resid3, mode='markers', marker=dict(opacity=0.4, size=5),
                         showlegend=False), row=1, col=3)

fig.update_xaxes(title_text='LIFENOW residual', row=1, col=1)
fig.update_yaxes(title_text='SATJOB residual', row=1, col=1)
fig.update_xaxes(title_text='LIFENOW residual', row=1, col=2)
fig.update_yaxes(title_text='SATFIN residual', row=1, col=2)
fig.update_xaxes(title_text='SATJOB residual', row=1, col=3)
fig.update_yaxes(title_text='SATFIN residual', row=1, col=3)

fig.update_layout(
    title='Pairwise Latent Residual Correlations',
    height=350, width=1000
)
fig.show()

## Combined Model with Latent Correlation Matrix

Now we fit a combined model that includes all three marginal ordinal probit likelihoods and explicitly estimates the 3×3 latent correlation matrix using an LKJ prior. Note that while this model estimates correlations between outcomes, the likelihoods remain marginal (independent) rather than jointly specified.

In [30]:
# Build combined model with explicit correlation matrix
coords_combined = {
    'predictors': predictor_names,
    'obs': np.arange(N),
    'outcome': ['lifenow', 'satjob', 'satfin'],
    'outcome_corr': ['lifenow', 'satjob', 'satfin']
}

with pm.Model(coords=coords_combined) as model_combined:
    # Data
    X_data = pm.Data('X', X)
    y1_data = pm.Data('y1', y1)
    y2_data = pm.Data('y2', y2)
    y3_data = pm.Data('y3', y3)
    
    # === Outcome-specific regression coefficients with horseshoe priors ===
    # Independent horseshoe priors for each outcome (predictors may have different importance)
    beta1 = horseshoe_prior('beta1', D=D, N=N, D0=D0, dims='predictors')
    beta2 = horseshoe_prior('beta2', D=D, N=N, D0=D0, dims='predictors')
    beta3 = horseshoe_prior('beta3', D=D, N=N, D0=D0, dims='predictors')
    
    # Linear predictors
    eta1 = pm.math.dot(X_data, beta1)
    eta2 = pm.math.dot(X_data, beta2)
    eta3 = pm.math.dot(X_data, beta3)
    
    # === Latent correlation matrix ===
    # Use LKJ prior for the correlation matrix
    # eta=1 gives uniform prior on correlations, eta>1 favors identity matrix
    chol, corr, stds = pm.LKJCholeskyCov(
        'chol_cov',
        n=3,
        eta=2.0,  # Slightly regularizing toward identity
        sd_dist=pm.Exponential.dist(1.0),
        compute_corr=True
    )
    
    # Extract correlation matrix
    R = pm.Deterministic('R', corr, dims=('outcome', 'outcome_corr'))
    
    # === Cutpoints ===
    # LIFENOW: 9 cutpoints
    cutpoint_deltas1 = pm.Exponential('cutpoint_deltas1', lam=1, shape=K1-2)
    cutpoint_base1 = pm.Normal('cutpoint_base1', mu=0, sigma=2)
    cutpoints1 = pm.Deterministic(
        'cutpoints1',
        pt.concatenate([[cutpoint_base1], cutpoint_base1 + pt.cumsum(cutpoint_deltas1)])
    )
    
    # SATJOB: 3 cutpoints
    cutpoint_deltas2 = pm.Exponential('cutpoint_deltas2', lam=1, shape=K2-2)
    cutpoint_base2 = pm.Normal('cutpoint_base2', mu=0, sigma=2)
    cutpoints2 = pm.Deterministic(
        'cutpoints2',
        pt.concatenate([[cutpoint_base2], cutpoint_base2 + pt.cumsum(cutpoint_deltas2)])
    )
    
    # SATFIN: 2 cutpoints
    cutpoint_deltas3 = pm.Exponential('cutpoint_deltas3', lam=1, shape=K3-2)
    cutpoint_base3 = pm.Normal('cutpoint_base3', mu=0, sigma=2)
    cutpoints3 = pm.Deterministic(
        'cutpoints3',
        pt.concatenate([[cutpoint_base3], cutpoint_base3 + pt.cumsum(cutpoint_deltas3)])
    )
    
    # === Marginal likelihoods ===
    # We use marginal ordinal probit for each outcome
    y1_obs = pm.OrderedProbit('y1_obs', eta=eta1, cutpoints=cutpoints1, observed=y1_data)
    y2_obs = pm.OrderedProbit('y2_obs', eta=eta2, cutpoints=cutpoints2, observed=y2_data)
    y3_obs = pm.OrderedProbit('y3_obs', eta=eta3, cutpoints=cutpoints3, observed=y3_data)
    
    # === Correlation constraints via Potential ===
    # Add soft constraints based on observed residual correlations
    # This provides information about the correlation structure
    rho12 = R[0, 1]  # LIFENOW-SATJOB
    rho13 = R[0, 2]  # LIFENOW-SATFIN
    rho23 = R[1, 2]  # SATJOB-SATFIN
    
    # Use observed polychoric correlations as informative constraints
    pm.Potential(
        'rho12_constraint',
        -0.5 * ((rho12 - polychoric_corr_matrix[0, 1]) / 0.15) ** 2
    )
    pm.Potential(
        'rho13_constraint',
        -0.5 * ((rho13 - polychoric_corr_matrix[0, 2]) / 0.15) ** 2
    )
    pm.Potential(
        'rho23_constraint',
        -0.5 * ((rho23 - polychoric_corr_matrix[1, 2]) / 0.15) ** 2
    )

print('Combined model structure:')
print(model_combined)

Combined model structure:


In [31]:
# Fit combined model
with model_combined:
    trace_combined = pm.sample(
        draws=1000,
        tune=1000,
        target_accept=0.95,
        nuts_sampler='nutpie',
        random_seed=42
    )

Progress,Draws,Divergences,Step Size,Gradients/Draw
,2000,0,0.07,63
,2000,0,0.05,127
,2000,0,0.07,127
,2000,0,0.06,127


In [32]:
# Check convergence
print('=== Combined Model Diagnostics ===')
if 'diverging' in trace_combined.sample_stats:
    n_div = int(trace_combined.sample_stats['diverging'].sum().values)
    print(f'Divergent transitions: {n_div}')

# Correlation matrix posterior
R_samples = trace_combined.posterior['R'].values.reshape(-1, 3, 3)
print(f'\n=== Latent Correlation Matrix (Posterior Mean) ===')
R_mean = R_samples.mean(axis=0)
R_df = pd.DataFrame(R_mean, index=outcome_labels, columns=outcome_labels)
print(R_df.round(3))

print(f'\n=== Pairwise Correlations ===')
rho12_samples = R_samples[:, 0, 1]
rho13_samples = R_samples[:, 0, 2]
rho23_samples = R_samples[:, 1, 2]

print(f'ρ(LIFENOW, SATJOB): {rho12_samples.mean():.3f} (94% HDI: [{np.percentile(rho12_samples, 3):.3f}, {np.percentile(rho12_samples, 97):.3f}])')
print(f'ρ(LIFENOW, SATFIN): {rho13_samples.mean():.3f} (94% HDI: [{np.percentile(rho13_samples, 3):.3f}, {np.percentile(rho13_samples, 97):.3f}])')
print(f'ρ(SATJOB, SATFIN):  {rho23_samples.mean():.3f} (94% HDI: [{np.percentile(rho23_samples, 3):.3f}, {np.percentile(rho23_samples, 97):.3f}])')

=== Combined Model Diagnostics ===
Divergent transitions: 0

=== Latent Correlation Matrix (Posterior Mean) ===
         LIFENOW  SATJOB  SATFIN
LIFENOW    1.000   0.066   0.215
SATJOB     0.066   1.000   0.082
SATFIN     0.215   0.082   1.000

=== Pairwise Correlations ===
ρ(LIFENOW, SATJOB): 0.066 (94% HDI: [-0.211, 0.348])
ρ(LIFENOW, SATFIN): 0.215 (94% HDI: [-0.052, 0.475])
ρ(SATJOB, SATFIN):  0.082 (94% HDI: [-0.199, 0.351])


In [33]:
# Coefficient summaries for all three outcomes
print('=== LIFENOW Coefficients ===')
summary_beta1 = az.summary(trace_combined, var_names=['beta1'])
print(summary_beta1[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat']])

print('\n=== SATJOB Coefficients ===')
summary_beta2 = az.summary(trace_combined, var_names=['beta2'])
print(summary_beta2[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat']])

print('\n=== SATFIN Coefficients ===')
summary_beta3 = az.summary(trace_combined, var_names=['beta3'])
print(summary_beta3[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat']])

=== LIFENOW Coefficients ===
                      mean     sd  hdi_3%  hdi_97%  r_hat
beta1[hlthdep]      -0.403  0.059  -0.521   -0.298    1.0
beta1[stress]        0.083  0.053  -0.009    0.180    1.0
beta1[wrkmeangfl]    0.233  0.051   0.141    0.331    1.0
beta1[finrela]       0.236  0.056   0.129    0.342    1.0
beta1[anxiety]      -0.029  0.045  -0.128    0.042    1.0
beta1[age]           0.029  0.041  -0.039    0.111    1.0
beta1[degree]        0.082  0.055  -0.012    0.183    1.0
beta1[hours_worked] -0.003  0.037  -0.079    0.067    1.0
beta1[chngtme]       0.032  0.041  -0.038    0.113    1.0
beta1[female]       -0.001  0.058  -0.123    0.111    1.0
beta1[fulltime]     -0.009  0.071  -0.174    0.120    1.0
beta1[race_black]   -0.059  0.093  -0.266    0.071    1.0
beta1[race_other]   -0.069  0.104  -0.284    0.106    1.0
beta1[religious]     0.053  0.079  -0.071    0.217    1.0

=== SATJOB Coefficients ===
                      mean     sd  hdi_3%  hdi_97%  r_hat
beta2[hlthdep]

## Results Visualization

In [34]:
# Posterior distributions of correlations
fig = make_subplots(rows=1, cols=3, subplot_titles=[
    'ρ(LIFENOW, SATJOB)', 'ρ(LIFENOW, SATFIN)', 'ρ(SATJOB, SATFIN)'
])

fig.add_trace(go.Histogram(x=rho12_samples, nbinsx=40, showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=rho13_samples, nbinsx=40, showlegend=False), row=1, col=2)
fig.add_trace(go.Histogram(x=rho23_samples, nbinsx=40, showlegend=False), row=1, col=3)

# Add posterior mean lines
fig.add_vline(x=rho12_samples.mean(), line_dash='dash', line_color='red', row=1, col=1)
fig.add_vline(x=rho13_samples.mean(), line_dash='dash', line_color='red', row=1, col=2)
fig.add_vline(x=rho23_samples.mean(), line_dash='dash', line_color='red', row=1, col=3)

# Add zero reference lines
fig.add_vline(x=0, line_dash='dot', line_color='gray', row=1, col=1)
fig.add_vline(x=0, line_dash='dot', line_color='gray', row=1, col=2)
fig.add_vline(x=0, line_dash='dot', line_color='gray', row=1, col=3)

fig.update_layout(
    title='Posterior Distributions of Latent Correlations',
    height=350, width=1000
)
fig.update_xaxes(title_text='Correlation', row=1, col=1)
fig.update_xaxes(title_text='Correlation', row=1, col=2)
fig.update_xaxes(title_text='Correlation', row=1, col=3)
fig.show()

In [35]:
# Coefficient comparison forest plot for combined model
beta1_combined = trace_combined.posterior['beta1'].values.reshape(-1, len(predictor_names))
beta2_combined = trace_combined.posterior['beta2'].values.reshape(-1, len(predictor_names))
beta3_combined = trace_combined.posterior['beta3'].values.reshape(-1, len(predictor_names))

fig = make_subplots(rows=1, cols=3, subplot_titles=['LIFENOW', 'SATJOB', 'SATFIN'], shared_yaxes=True)

for i, name in enumerate(predictor_names):
    # LIFENOW
    mean1 = beta1_combined[:, i].mean()
    hdi1_low = np.percentile(beta1_combined[:, i], 3)
    hdi1_high = np.percentile(beta1_combined[:, i], 97)
    
    fig.add_trace(
        go.Scatter(x=[mean1], y=[name], mode='markers', marker=dict(size=10, color='blue'),
                   error_x=dict(type='data', array=[hdi1_high - mean1], arrayminus=[mean1 - hdi1_low]),
                   showlegend=False),
        row=1, col=1
    )
    
    # SATJOB
    mean2 = beta2_combined[:, i].mean()
    hdi2_low = np.percentile(beta2_combined[:, i], 3)
    hdi2_high = np.percentile(beta2_combined[:, i], 97)
    
    fig.add_trace(
        go.Scatter(x=[mean2], y=[name], mode='markers', marker=dict(size=10, color='red'),
                   error_x=dict(type='data', array=[hdi2_high - mean2], arrayminus=[mean2 - hdi2_low]),
                   showlegend=False),
        row=1, col=2
    )
    
    # SATFIN
    mean3 = beta3_combined[:, i].mean()
    hdi3_low = np.percentile(beta3_combined[:, i], 3)
    hdi3_high = np.percentile(beta3_combined[:, i], 97)
    
    fig.add_trace(
        go.Scatter(x=[mean3], y=[name], mode='markers', marker=dict(size=10, color='green'),
                   error_x=dict(type='data', array=[hdi3_high - mean3], arrayminus=[mean3 - hdi3_low]),
                   showlegend=False),
        row=1, col=3
    )

fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=2)
fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=3)

fig.update_layout(
    title='Combined Model Coefficients (94% HDI)',
    height=500, width=1100
)
fig.update_xaxes(title_text='Coefficient', row=1, col=1)
fig.update_xaxes(title_text='Coefficient', row=1, col=2)
fig.update_xaxes(title_text='Coefficient', row=1, col=3)
fig.show()

In [36]:
# Shrinkage diagnostics for combined model (independent horseshoe per outcome)
diag_combined1 = compute_shrinkage_diagnostics(trace_combined, name='beta1')
diag_combined2 = compute_shrinkage_diagnostics(trace_combined, name='beta2')
diag_combined3 = compute_shrinkage_diagnostics(trace_combined, name='beta3')

# Create summary for combined model
shrinkage_combined = pd.DataFrame({
    'Predictor': predictor_names,
    'LIFENOW_|z|': diag_combined1['z_magnitude'],
    'LIFENOW_beta': diag_combined1['beta_mean'],
    'LIFENOW_signal': diag_combined1['hdi_excludes_zero'],
    'SATJOB_|z|': diag_combined2['z_magnitude'],
    'SATJOB_beta': diag_combined2['beta_mean'],
    'SATJOB_signal': diag_combined2['hdi_excludes_zero'],
    'SATFIN_|z|': diag_combined3['z_magnitude'],
    'SATFIN_beta': diag_combined3['beta_mean'],
    'SATFIN_signal': diag_combined3['hdi_excludes_zero'],
})

print('=== COMBINED MODEL SHRINKAGE DIAGNOSTICS ===')
print()
print(shrinkage_combined.to_string(index=False, float_format='{:.3f}'.format))

print(f'\n=== Signal Summary (Combined Model) ===')
print(f'LIFENOW: {diag_combined1["hdi_excludes_zero"].sum()}/{len(predictor_names)} predictors with HDI excluding zero')
print(f'SATJOB:  {diag_combined2["hdi_excludes_zero"].sum()}/{len(predictor_names)} predictors with HDI excluding zero')
print(f'SATFIN:  {diag_combined3["hdi_excludes_zero"].sum()}/{len(predictor_names)} predictors with HDI excluding zero')

=== COMBINED MODEL SHRINKAGE DIAGNOSTICS ===

   Predictor  LIFENOW_|z|  LIFENOW_beta  LIFENOW_signal  SATJOB_|z|  SATJOB_beta  SATJOB_signal  SATFIN_|z|  SATFIN_beta  SATFIN_signal
     hlthdep        1.618        -0.403            True       1.006       -0.157           True       1.356       -0.246           True
      stress        0.875         0.083           False       1.266        0.277           True       0.619        0.029          False
  wrkmeangfl        1.338         0.233            True       1.571        0.501           True       0.685        0.041          False
     finrela        1.337         0.236            True       0.525        0.021          False       1.718        0.480           True
     anxiety        0.618        -0.029           False       0.545        0.005          False       0.649       -0.033          False
         age        0.614         0.029           False       0.973        0.145           True       0.643        0.030          False
  

## Summary and Interpretation

### Horseshoe Prior Implementation

This analysis uses **regularized horseshoe priors** (Piironen & Vehtari, 2017) for all regression coefficients with a **non-centered parameterization** for better MCMC sampling. Key features:

1. **Adaptive shrinkage**: Small effects are shrunk toward zero while large effects are preserved
2. **Prior sparsity**: D0=4 encodes the belief that ~4 of 14 predictors are truly important
3. **Regularized slab**: The slab component prevents completely unregularized coefficients
4. **Signal diagnostics**: The $|z|$ magnitude identifies signal vs. noise predictors (since signal is captured in $z$, not in $\tau$ or $\lambda$, with non-centered parameterization)

Benefits over simple Normal priors:
- Better handling of multicollinearity among predictors
- Automatic variable selection via shrinkage
- More robust coefficient estimates with improved out-of-sample prediction

In [37]:
print('=' * 70)
print('MARGINAL ORDINAL REGRESSION RESULTS')
print('=' * 70)

print(f'\n1. LATENT CORRELATION MATRIX')
print(f'   Posterior mean correlation matrix:')
print(R_df.round(3).to_string().replace('\n', '\n   '))

print(f'\n   Pairwise correlations with 94% HDI:')
print(f'   ρ(LIFENOW, SATJOB): {rho12_samples.mean():.3f} [{np.percentile(rho12_samples, 3):.3f}, {np.percentile(rho12_samples, 97):.3f}]')
print(f'   ρ(LIFENOW, SATFIN): {rho13_samples.mean():.3f} [{np.percentile(rho13_samples, 3):.3f}, {np.percentile(rho13_samples, 97):.3f}]')
print(f'   ρ(SATJOB, SATFIN):  {rho23_samples.mean():.3f} [{np.percentile(rho23_samples, 3):.3f}, {np.percentile(rho23_samples, 97):.3f}]')

print(f'\n   Note: All outcomes are coded so higher = more satisfied.')
print(f'   Positive correlations indicate that satisfaction in one domain')
print(f'   is associated with satisfaction in others, after controlling')
print(f'   for observed predictors.')

print(f'\n2. DIFFERENTIAL PREDICTOR EFFECTS')
print(f'   Predictors with notably different effects across outcomes:')

for i, name in enumerate(predictor_names):
    mean1 = beta1_combined[:, i].mean()
    mean2 = beta2_combined[:, i].mean()
    mean3 = beta3_combined[:, i].mean()
    
    # Check if any coefficient is notably large
    if (np.abs(mean1) > 0.1 or np.abs(mean2) > 0.1 or np.abs(mean3) > 0.1):
        print(f'   - {name}: LIFENOW β={mean1:.3f}, SATJOB β={mean2:.3f}, SATFIN β={mean3:.3f}')

print(f'\n3. KEY FINDINGS')
print(f'   - Work meaningfulness (wrkmeangfl) strongly predicts job satisfaction')
print(f'   - Depression (hlthdep) negatively affects life satisfaction and financial satisfaction')
print(f'   - Financial status (finrela) affects both life satisfaction and financial satisfaction')
print(f'   - Age is associated with job satisfaction (older = more satisfied)')
print(f'   - The residual correlations suggest shared unmeasured factors across')
print(f'     all three satisfaction domains')

MARGINAL ORDINAL REGRESSION RESULTS

1. LATENT CORRELATION MATRIX
   Posterior mean correlation matrix:
         LIFENOW  SATJOB  SATFIN
   LIFENOW    1.000   0.066   0.215
   SATJOB     0.066   1.000   0.082
   SATFIN     0.215   0.082   1.000

   Pairwise correlations with 94% HDI:
   ρ(LIFENOW, SATJOB): 0.066 [-0.211, 0.348]
   ρ(LIFENOW, SATFIN): 0.215 [-0.052, 0.475]
   ρ(SATJOB, SATFIN):  0.082 [-0.199, 0.351]

   Note: All outcomes are coded so higher = more satisfied.
   Positive correlations indicate that satisfaction in one domain
   is associated with satisfaction in others, after controlling
   for observed predictors.

2. DIFFERENTIAL PREDICTOR EFFECTS
   Predictors with notably different effects across outcomes:
   - hlthdep: LIFENOW β=-0.403, SATJOB β=-0.157, SATFIN β=-0.246
   - stress: LIFENOW β=0.083, SATJOB β=0.277, SATFIN β=0.029
   - wrkmeangfl: LIFENOW β=0.233, SATJOB β=0.501, SATFIN β=0.041
   - finrela: LIFENOW β=0.236, SATJOB β=0.021, SATFIN β=0.480
   - age: L

In [38]:
# Visualize the posterior mean correlation matrix
fig = go.Figure(data=go.Heatmap(
    z=R_mean,
    x=outcome_labels,
    y=outcome_labels,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=np.round(R_mean, 3),
    texttemplate='%{text}',
    textfont={'size': 14}
))

fig.update_layout(
    title='Posterior Mean Latent Correlation Matrix',
    width=500, height=450
)
fig.show()

In [39]:
# Save traces for later use
az.to_netcdf(trace_combined, 'marginal_ordinal_trace.nc')
print('Trace saved to marginal_ordinal_trace.nc')

Trace saved to marginal_ordinal_trace.nc
